# 06 - Final project story: data, model, and GIS outputs

This read-only notebook presents the completed capstone from data design through final temporal model evidence and GIS context. It uses only validated project artefacts. It does not fit a model, calculate a new score, rewrite an image, create a forecast, or make a purchase recommendation.

The spatial context figures describe **1 km mainland grid cells with fire recurrence measured in a 2 km context**.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Repository root resolved; all paths below are repository-relative.')

## 1. Project design and evidence contract

The model estimates continuous next-year burned share from predictor-year information only. The final temporal test remains separate from model selection.

In [ ]:
from src.feature_contract import FIELD_CONTRACTS, PREDICTOR_COLUMNS, TARGET_COLUMN
from src.final_visuals import validate_final_visuals
from src.model_diagnostics import validate_model_diagnostics

national_validation_path = PROJECT_ROOT / 'data/processed/national_panel_2015_2024_validation.json'
eda_path = PROJECT_ROOT / 'reports/validation/national_panel_model_readiness_eda.json'
final_metrics_path = PROJECT_ROOT / 'data/processed/extended_model_selection_2010_2021/final_temporal_test_metrics.json'
national_validation = json.loads(national_validation_path.read_text(encoding='utf-8'))
eda = json.loads(eda_path.read_text(encoding='utf-8'))
final_metrics = json.loads(final_metrics_path.read_text(encoding='utf-8'))
presentation_validation = validate_final_visuals()
diagnostic_inventory = validate_model_diagnostics()

assert len(PREDICTOR_COLUMNS) == 9
assert national_validation['actual_row_count'] == 891_120
assert tuple(final_metrics['design']['final_test_years']) == (2022, 2023, 2024)
assert diagnostic_inventory['status'] == 'verified_existing'

design = pd.DataFrame([
    ('Spatial unit', 'One EPSG:3763 mainland Portugal 1 km cell per predictor year'),
    ('Panel', f"{national_validation['grid_cell_count']:,} cells; {national_validation['actual_row_count']:,} cell-year rows for T=2015-2024"),
    ('Predictors', f"{len(PREDICTOR_COLUMNS)} leakage-safe predictors"),
    ('Target', 'Continuous burned_share_next_year in T+1'),
    ('Frozen model training', 'T=2010-2019'),
    ('Model validation', 'T=2020-2021'),
    ('Final temporal test', 'T=2022-2024; outcomes 2023-2025'),
], columns=['Item', 'Validated design'])
display(design)

## 2. Nine-predictor data-science contract

All predictors are present in the final model in this exact order. The source-year rule makes the temporal boundary auditable.

In [ ]:
feature_contract = pd.DataFrame([
    {
        'Predictor': name,
        'Unit': FIELD_CONTRACTS[name].unit,
        'Allowed range': f"{FIELD_CONTRACTS[name].minimum} to {FIELD_CONTRACTS[name].maximum}",
        'Temporal/source rule': FIELD_CONTRACTS[name].source_year_rule,
    }
    for name in PREDICTOR_COLUMNS
])
display(feature_contract)
print('Target:', TARGET_COLUMN)

## 3. What the data shows before modelling

The target is continuous but strongly zero-heavy, so all-row error alone is not enough. Correlations are screening evidence for redundancy, not causal relationships.

In [ ]:
eda_figures = {
    'Target distribution by predictor year': 'reports/figures/panel_eda_target_by_year.png',
    'Predictor correlation matrix': 'reports/figures/panel_eda_predictor_correlations.png',
}
for title, relative_path in eda_figures.items():
    path = PROJECT_ROOT / relative_path
    assert path.is_file() and path.stat().st_size > 5_000
    print(title, '<-', relative_path)
    display(Image(filename=str(path), width=900))
print(f"Overall zero-target proportion: {eda['target']['overall_zero_proportion']:.2%}")

## 4. Final temporal model evidence

The retained nine-feature hurdle estimates expected next-year burned share as an occurrence component multiplied by a conditional positive-share component. It is compared with a training-only historical-recurrence baseline. These are regression estimates, not probabilities.

In [ ]:
overall_metrics_path = PROJECT_ROOT / diagnostic_inventory['tables']['overall_metrics']
by_year_metrics_path = PROJECT_ROOT / diagnostic_inventory['tables']['by_year_metrics']
overall_metrics = pd.read_csv(overall_metrics_path)
by_year_metrics = pd.read_csv(by_year_metrics_path)
model_labels = {
    'historical_recurrence_baseline': 'Historical recurrence baseline',
    'nine_feature_hurdle': 'Nine-feature hurdle',
}
for frame in (overall_metrics, by_year_metrics):
    frame['model'] = frame['model'].map(model_labels).fillna(frame['model'])

display(overall_metrics[['model', 'MAE', 'RMSE', 'positive_row_MAE', 'capture_at_20_percent']])
display(by_year_metrics[['predictor_year', 'model', 'MAE', 'RMSE', 'positive_row_MAE', 'capture_at_20_percent']])
print('MAE and RMSE: lower is better. Positive-row MAE focuses on cells that burned. Capture@20% is a technical ranking diagnostic, not a buyer threshold.')

In [ ]:
for title, relative_path in diagnostic_inventory['figures'].items():
    path = PROJECT_ROOT / relative_path
    assert path.is_file() and path.stat().st_size > 5_000
    print(title.replace('_', ' ').title(), '<-', relative_path)
    display(Image(filename=str(path), width=900))

## 5. GIS context: historical evidence and official comparison

The following presentation layer is separate from the regression output. It is historical comparative exposure based on 2016-2025 recurrence in a 2 km context, shown beside official ICNF structural hazard for context.

In [ ]:
spatial_figure_names = [
    'historical_exposure_map',
    'historical_icnf_comparison_map',
    'crosstab',
    'summary_table',
]
for name in spatial_figure_names:
    record = presentation_validation['figures'][name]
    path = PROJECT_ROOT / record['path']
    assert record['status'] == 'verified_existing' and path.is_file()
    print(name.replace('_', ' ').title(), '<-', record['source_data'])
    display(Image(filename=str(path), width=900))

## 6. Correct use, limitations, and reproducibility

The final map and model evidence support broad-area comparative research only. They are not a next-year forecast guarantee, a property-level safety assessment, or a purchase recommendation.

In [ ]:
record = presentation_validation['figures']['decision_limitations']
path = PROJECT_ROOT / record['path']
assert record['status'] == 'verified_existing' and path.is_file()
display(Image(filename=str(path), width=900))

print('QGIS project: qgis/wildfire_exposure_screening_portugal.qgz')
print('Historical screening GeoPackage: data/processed/spatial_outputs/historical_residential_wildfire_exposure_screening.gpkg')
print('Full reproducible rebuild: python scripts/run_project.py --mode reproduce --confirm-rebuild')

## Optional controlled regeneration

Normal execution only verifies and displays outputs. Set the switch below to `True` only to rebuild the four code-generated historical-screening visuals from their validated sources. Model diagnostics have their own deliberate regeneration command in `scripts/build_model_diagnostics.py`; QGIS maps remain a separate explicit QGIS step.

In [ ]:
REGENERATE_FINAL_VISUALS = False
if REGENERATE_FINAL_VISUALS:
    from src.final_visuals import build_final_visuals
    rebuilt_visuals = build_final_visuals()
    print('Regenerated historical-screening visuals:', rebuilt_visuals)
else:
    print('Read-only presentation mode. No images or model artefacts were rewritten.')